# 02 — Pipeline Preprocessing

Berdasarkan temuan EDA (`01_eda.ipynb`), pipeline ini menerapkan:

| # | Langkah | Alasan |
|---|---|---|
| 1 | **Lowercase** | Semua teks uppercase/mixed ke huruf kecil |
| 2 | **Hapus URL** | Noise, tidak informatif |
| 3 | **Hapus Mention & Hashtag** | Noise |
| 4 | **Hapus Emoji** | 108 teks mengandung emoji |
| 5 | **Hapus Angka** | 4.257 teks mengandung angka |
| 6 | **Hapus Tanda Baca** | Noise untuk TF-IDF & embedding |
| 7 | **Normalisasi Slang** | Banyak kata informal Indonesia |
| 8 | **Min token length = 2** | Filter token terlalu pendek |

**Output:**
- `../data/processed/cleaned_text.csv` → digunakan `04_deep_learning.ipynb`
- `../models/model_ml/pipeline.pkl` → digunakan `03_pycaret_model.ipynb`


In [8]:
from __future__ import annotations
import re, pickle, logging, os
from pathlib import Path
from typing import List, Optional, Union
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer as _SklearnTFIDF

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(asctime)s — %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)
print('✅ Library siap')

✅ Library siap


In [9]:
class _PreprocessingUnpickler(pickle.Unpickler):
    _LOCAL_CLASSES = {'TextPreprocessor', 'TFIDFVectorizer', 'PreprocessingPipeline'}
    def find_class(self, module, name):
        import importlib
        if module == '__main__' and name in self._LOCAL_CLASSES:
            try:
                return getattr(importlib.import_module('preprocessing'), name)
            except (ImportError, AttributeError):
                return getattr(importlib.import_module(__name__), name)
        return super().find_class(module, name)

In [10]:
class TextPreprocessor:
    """
    Membersihkan teks Bahasa Indonesia informal/slang.

    Tahapan (sesuai temuan EDA):
      Lowercase → Hapus URL → Mention/Hashtag → Emoji → Angka → Tanda Baca → Slang → Filter token pendek
    """
    # Unicode ranges emoji & simbol khusus
    _EMOJI_PATTERN = re.compile(
        u'['
        u'\U0001F300-\U0001F9FF'  # Misc symbols, emoticons, transport, etc.
        u'\U0001FA00-\U0001FA6F'  # Chess symbols, etc.
        u'\U0001FA70-\U0001FAFF'  # Symbols and pictographs extended-A
        u'\U00002702-\U000027B0'  # Dingbats
        u'\U000024C2-\U0001F251'  # Enclosed chars
        u']+',
        flags=re.UNICODE
    )

    def __init__(
        self,
        lowercase=True,
        remove_url=True,
        remove_mention_hashtag=True,
        remove_emoji=True,           # ← BARU: hapus emoji (108 teks, temuan EDA)
        remove_punctuation=True,
        remove_numbers=True,         # ← 4.257 teks mengandung angka
        min_token_length=2,          # ← 104 teks sangat pendek (<3 kata)
        slang_dict=None
    ):
        self.lowercase              = lowercase
        self.remove_url             = remove_url
        self.remove_mention_hashtag = remove_mention_hashtag
        self.remove_emoji           = remove_emoji
        self.remove_punctuation     = remove_punctuation
        self.remove_numbers         = remove_numbers
        self.min_token_length       = min_token_length

        self._url_re        = re.compile(r'https?://\S+|www\.\S+')
        self._mention_re    = re.compile(r'@\w+')
        self._hashtag_re    = re.compile(r'#\w+')
        self._punct_re      = re.compile(r'[^\w\s]')
        self._number_re     = re.compile(r'\d+')
        self._whitespace_re = re.compile(r'\s+')

        self.slang_mapping = {}
        if isinstance(slang_dict, dict):
            self.slang_mapping = slang_dict
        elif slang_dict is not None:
            self._load_slang_csv(slang_dict)

    def _load_slang_csv(self, path):
        path = Path(path)
        if not path.exists():
            logger.warning('File slang tidak ditemukan: %s', path.resolve()); return
        try:
            df = pd.read_csv(path)
            if 'slang' in df.columns and 'formal' in df.columns:
                df = df.dropna(subset=['formal'])
                self.slang_mapping = dict(zip(df['slang'], df['formal']))
                logger.info('Berhasil memuat %d kata slang dari %s', len(self.slang_mapping), path.name)
        except Exception as e:
            logger.error('Gagal memuat file slang: %s', e)

    def clean(self, text):
        if not isinstance(text, str): text = str(text)
        if self.lowercase:              text = text.lower()
        if self.remove_url:             text = self._url_re.sub(' ', text)
        if self.remove_mention_hashtag:
            text = self._mention_re.sub(' ', text)
            text = self._hashtag_re.sub(' ', text)
        if self.remove_emoji:           text = self._EMOJI_PATTERN.sub(' ', text)  # ← BARU
        if self.remove_numbers:         text = self._number_re.sub('', text)        # angka dihapus dulu
        if self.remove_punctuation:     text = self._punct_re.sub(' ', text)
        tokens = self._whitespace_re.split(text.strip())
        return ' '.join(
            str(self.slang_mapping.get(t, t))
            for t in tokens
            if len(t) >= self.min_token_length
        )

    def transform(self, texts):
        logger.info('TextPreprocessor: membersihkan %d teks...', len(texts))
        cleaned = [self.clean(t) for t in texts]
        logger.info('TextPreprocessor: selesai.')
        return cleaned

    def __repr__(self):
        return (f'TextPreprocessor(lowercase={self.lowercase}, '
                f'remove_emoji={self.remove_emoji}, min_token_length={self.min_token_length})')

In [11]:
class TFIDFVectorizer:
    """
    Wrapper TF-IDF dengan default dioptimalkan untuk teks Indonesia informal.
    Config: max_features=5000, ngram_range=(1,2), min_df=3, max_df=0.90, sublinear_tf=True
    """
    def __init__(self, max_features=5000, ngram_range=(1,2), min_df=3, max_df=0.90, sublinear_tf=True):
        self.max_features = max_features
        self.ngram_range  = ngram_range
        self.min_df       = min_df
        self.max_df       = max_df
        self.sublinear_tf = sublinear_tf
        self._is_fitted   = False
        self._vectorizer  = _SklearnTFIDF(
            max_features=max_features, ngram_range=ngram_range,
            min_df=min_df, max_df=max_df, sublinear_tf=sublinear_tf, analyzer='word')

    def fit(self, texts):
        logger.info('TFIDFVectorizer: fitting pada %d teks...', len(texts))
        self._vectorizer.fit(texts)
        self._is_fitted = True
        logger.info('TFIDFVectorizer: selesai. Jumlah fitur = %d', len(self._vectorizer.get_feature_names_out()))
        return self

    def transform(self, texts):
        self._check_fitted()
        texts = [t if isinstance(t, str) else '' for t in texts]  # guard NaN
        matrix = self._vectorizer.transform(texts)
        return pd.DataFrame(matrix.toarray(), columns=self._vectorizer.get_feature_names_out())

    def fit_transform(self, texts):
        return self.fit(texts).transform(texts)

    @property
    def feature_names(self): self._check_fitted(); return list(self._vectorizer.get_feature_names_out())
    @property
    def n_features(self): return len(self.feature_names)
    def _check_fitted(self):
        if not self._is_fitted: raise RuntimeError('TFIDFVectorizer belum di-fit.')
    def __repr__(self):
        return f'TFIDFVectorizer(max_features={self.max_features}, ngram_range={self.ngram_range})'

In [12]:
class PreprocessingPipeline:
    """
    Pipeline lengkap: CSV/DataFrame → teks bersih → fitur TF-IDF → DataFrame siap PyCaret.
    """
    def __init__(self, text_col='review_text', target_col='sentiment',
                 preprocessor=None, vectorizer=None, slang_dict=None):
        self.text_col     = text_col
        self.target_col   = target_col
        self.slang_dict   = slang_dict
        self.preprocessor = preprocessor or TextPreprocessor(slang_dict=slang_dict)
        self.vectorizer   = vectorizer   or TFIDFVectorizer()
        self._is_fitted   = False

    def fit_transform(self, source, save_cleaned=False, cleaned_path='data/processed/cleaned_text.csv'):
        df = self._load(source)
        texts, labels = self._split_text_label(df)
        cleaned = self.preprocessor.transform(texts)
        if save_cleaned:
            Path(cleaned_path).parent.mkdir(parents=True, exist_ok=True)
            pd.DataFrame({'clean_text': cleaned, self.target_col: labels}).to_csv(cleaned_path, index=False)
            logger.info('Teks bersih disimpan ke %s', cleaned_path)
        cleaned_nonnull = [t if t else '' for t in cleaned]  # guard empty string
        X_df = self.vectorizer.fit_transform(cleaned_nonnull)
        X_df[self.target_col] = labels.reset_index(drop=True)
        self._is_fitted = True
        logger.info('Pipeline selesai. Output shape: %s | Fitur: %d', X_df.shape, self.vectorizer.n_features)
        return X_df

    def transform(self, source, include_label=True):
        self._check_fitted()
        df = self._load(source)
        cleaned = self.preprocessor.transform(df[self.text_col])
        X_df = self.vectorizer.transform(cleaned)
        if include_label and self.target_col in df.columns:
            X_df[self.target_col] = df[self.target_col].reset_index(drop=True)
        return X_df

    def transform_raw(self, texts):
        self._check_fitted()
        if isinstance(texts, str): texts = [texts]
        return self.vectorizer.transform(self.preprocessor.transform(texts))

    def save(self, path='pipeline.pkl'):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'wb') as f: pickle.dump(self, f)
        logger.info('Pipeline disimpan ke: %s', path.resolve())

    @classmethod
    def load(cls, path='pipeline.pkl'):
        path = Path(path)
        if not path.exists(): raise FileNotFoundError(f'File tidak ditemukan: {path}')
        with open(path, 'rb') as f: obj = _PreprocessingUnpickler(f).load()
        logger.info('Pipeline dimuat dari: %s', path.resolve())
        return obj

    @property
    def is_fitted(self): return self._is_fitted
    @property
    def feature_names(self): self._check_fitted(); return self.vectorizer.feature_names
    @property
    def n_features(self): self._check_fitted(); return self.vectorizer.n_features

    def summary(self):
        print('=' * 55)
        print('         PREPROCESSING PIPELINE SUMMARY')
        print('=' * 55)
        print(f'  Kolom teks   : {self.text_col}')
        print(f'  Kolom target : {self.target_col}')
        print(f'  Status       : {"✅ Fitted" if self._is_fitted else "⏳ Belum di-fit"}')
        print(f'  [TextPreprocessor]')
        print(f'    Hapus emoji    : {self.preprocessor.remove_emoji}')
        print(f'    Hapus angka    : {self.preprocessor.remove_numbers}')
        print(f'    Min token len  : {self.preprocessor.min_token_length}')
        print(f'    Kata slang     : {len(self.preprocessor.slang_mapping):,}')
        print(f'  [TFIDFVectorizer]')
        print(f'    Max features   : {self.vectorizer.max_features}')
        print(f'    N-gram range   : {self.vectorizer.ngram_range}')
        if self._is_fitted: print(f'    Fitur aktual   : {self.vectorizer.n_features}')
        print('=' * 55)

    def _load(self, source):
        if isinstance(source, pd.DataFrame): return source.copy()
        path = Path(source)
        if not path.exists(): raise FileNotFoundError(f'File tidak ditemukan: {path}')
        return pd.read_csv(path)

    def _split_text_label(self, df):
        if self.text_col not in df.columns: raise ValueError(f"Kolom '{self.text_col}' tidak ditemukan.")
        if self.target_col not in df.columns: raise ValueError(f"Kolom '{self.target_col}' tidak ditemukan.")
        return df[self.text_col], df[self.target_col]

    def _check_fitted(self):
        if not self._is_fitted: raise RuntimeError('Pipeline belum di-fit.')

    def __repr__(self):
        return f"PreprocessingPipeline(text_col='{self.text_col}', fitted={self._is_fitted})"

---
## Menjalankan Pipeline


In [13]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models/model_ml', exist_ok=True)

pipeline = PreprocessingPipeline(
    text_col='review_text',
    target_col='sentiment',
    slang_dict='../data/raw/slang-indo.csv'
)

df_tfidf = pipeline.fit_transform(
    '../data/raw/dataset.csv',
    save_cleaned=True,
    cleaned_path='../data/processed/cleaned_text.csv'
)

print('\n✅ Preprocessing selesai!')
print(f'   Shape TF-IDF  : {df_tfidf.shape}')
print(f'   Distribusi label:\n{df_tfidf["sentiment"].value_counts().to_string()}')
df_tfidf.head(3)

[INFO] 23:42:27 — Berhasil memuat 4408 kata slang dari slang-indo.csv
[INFO] 23:42:27 — TextPreprocessor: membersihkan 19728 teks...
[INFO] 23:42:28 — TextPreprocessor: selesai.
[INFO] 23:42:28 — Teks bersih disimpan ke ../data/processed/cleaned_text.csv
[INFO] 23:42:28 — TFIDFVectorizer: fitting pada 19728 teks...
[INFO] 23:42:30 — TFIDFVectorizer: selesai. Jumlah fitur = 5000
[INFO] 23:42:31 — Pipeline selesai. Output shape: (19728, 5001) | Fitur: 5000



✅ Preprocessing selesai!
   Shape TF-IDF  : (19728, 5001)
   Distribusi label:
sentiment
0    9864
1    9864


,abal,abal baru,abstrak,abu,acakan,acara,action,ada,ada bagian,ada bau,...,zonk habis,zonk menyesal,zonk parah,zonk sudah,zonk ternyata,zonk titik,zonk total,zzz,zzz enggak,sentiment
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.117601,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [14]:
pipeline.save('../models/model_ml/pipeline.pkl')

print('✅ Pipeline tersimpan!')
print('   Path : ../models/model_ml/pipeline.pkl')
pipeline.summary()

[INFO] 23:42:32 — Pipeline disimpan ke: C:\Razin\projects\sentiment-analysis (tubes-nlp)\models\model_ml\pipeline.pkl


✅ Pipeline tersimpan!
   Path : ../models/model_ml/pipeline.pkl
         PREPROCESSING PIPELINE SUMMARY
  Kolom teks   : review_text
  Kolom target : sentiment
  Status       : ✅ Fitted
  [TextPreprocessor]
    Hapus emoji    : True
    Hapus angka    : True
    Min token len  : 2
    Kata slang     : 4,408
  [TFIDFVectorizer]
    Max features   : 5000
    N-gram range   : (1, 2)
    Fitur aktual   : 5000
